# Trabalho final: Modelos de recomendação e análise de espaços latentes

**Disciplina:** *Aprendizado não-supervisionado* - *MC847*\
**Instituição:** *Universidade Estadual de Campinas*\
**Professor(a):** *Marcos Medeiros Raimundo*\
**Semestre:** *2s2026*

| Nome | RA |
|---|---|
| *Integrante 1* | *RA 1* |
| *Integrante 2* | *RA 2* |
| *Integrante 3 (opcional)* | *RA 3* |

Preencha a tabela acima com os nomes e RAs dos integrantes da equipe (2 ou 3 alunos, impreterivelmente).

## Introdução

Em sistemas de recomendação, a principal tarefa é aprender a preferência de um usuário a partir de um histórico parcial de interações.
Isso pode ser visto como um problema de *completar informação*: dado o histórico parcial de um usuário, quais itens provavelmente faltam?
Neste trabalho vocês vão prever notas não observadas com modelos que fazem suposições diferentes sobre essa estrutura e, depois, investigar o que o espaço aprendido contém e quais suposições o produziram.

O trabalho é dividido em três etapas, sendo as duas primeiras comuns a todos os grupos e a terceira a ser definida por cada grupo.
A primeira etapa consiste em definir uma baseline que será usada como referência para as demais etapas.
A segunda consiste em realizar e analisar experimentos para entender como diferentes suposições sobre a estrutura latente dos dados afetam o desempenho do modelo.
A última etapa é a contribuição de cada grupo para o problema, que pode ser uma proposta de melhoria do modelo ou uma análise mais profunda de algum aspecto do problema.

As duas primeiras etapas têm códigos de referência, que podem ser usados como base para a implementação dos modelos.
Vocês podem usar bibliotecas externas com implementações de modelos, mas devem deixar claro quais bibliotecas foram usadas e como elas foram usadas.
Por exemplo, vocês podem reescrever o modelo de fatoração de matrizes em PyTorch ou Jax para melhorar a performance, desde que justifiquem a escolha e expliquem como a implementação foi mantida fiel ao modelo original.

## Dataset

Utilizaremos o conjunto de dados MovieLens 100K [(Harper e Konstan, 2015)](https://www.scirp.org/reference/referencespapers?referenceid=3586299): 100 mil notas inteiras de 1 a 5, dadas por 943 usuários a 1.682 filmes, com pelo menos 20 notas por usuário.
A matriz tem densidade de cerca de 6%, cada usuário tem idade (em anos), gênero e ocupação (21 categorias) e cada filme tem título, ano e 19 indicadores de gênero, e um filme pode ter vários gêneros.

Devido à natureza do problema, a ausência de notas não pode ser tratada como aleatória: Os motivos pelos quais um usuário não avaliou um item podem ser variados, incluindo falta de interesse, falta de exposição ao item ou simplesmente esquecimento.
Isso significa que os dados de treinamento são enviesados e não representam a distribuição real de preferências dos usuários.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from numpy.typing import NDArray

sns.set_theme(style="whitegrid")

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
DATA_DIR = "./data/"

def set_seed(seed: int) -> None:
    """Set the random seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
from pandas.core.frame import DataFrame

GENRES = [
    "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"
]


def load_movielens() -> tuple[DataFrame, DataFrame, DataFrame]:
    """Download and extract the MovieLens dataset."""
    import zipfile

    import requests

    data_dir = Path(DATA_DIR)
    data_dir.mkdir(parents=True, exist_ok=True)

    zip_path = data_dir / "ml-100k.zip"
    target_dir = data_dir / "ml-100k"

    if not target_dir.exists():
        if not zip_path.exists():
            print("Downloading MovieLens dataset...")
            response = requests.get(MOVIELENS_URL)

            with open(zip_path, "wb") as f:
                f.write(response.content)

            print("Download complete.")

        print("Extracting MovieLens dataset...")
        with zipfile.ZipFile(zip_path) as zip_ref:
            zip_ref.extractall(data_dir)

        print(f"Extracted to {data_dir}.")

    ratings = pd.read_csv(
            target_dir / "u.data",
            sep="\t",
            names=["user_id", "item_id", "rating", "timestamp"],
            dtype={"user_id": np.int32, "item_id": np.int32, "rating": np.float32, "timestamp": np.int64},
        )

    users = pd.read_csv(
            target_dir / "u.user",
            sep="|",
            names=["user_id", "age", "gender", "occupation", "zip_code"],
            dtype={"user_id": np.int32, "age": np.int32, "gender": str, "occupation": str, "zip_code": str},
        )

    item_cols = ["item_id", "title", "release_date", "video_release_date", "imdb_url", *GENRES]

    movies = pd.read_csv(
        target_dir / "u.item",
        sep="|",
        encoding="latin-1",
        names=item_cols,
        usecols=["item_id", "title", "release_date", *GENRES],
        dtype={"item_id": np.int32, "title": str, "release_date": str},
    )

    movies["year"] = pd.to_numeric(movies["title"].str.extract(r"\((\d{4})\)")[0], errors="coerce")

    movies["genres"] = movies[GENRES].apply(
        lambda row: "|".join([g for g in GENRES if row[g] == 1]), axis=1
    )

    # Normalização para que os IDs de usuário e item comecem em 0
    ratings["user_id"] -= 1
    ratings["item_id"] -= 1
    users["user_id"] -= 1
    movies["item_id"] -= 1

    return ratings, users, movies


def ratings_to_matrix(ratings: pd.DataFrame, fill_value: float = float("nan")) -> NDArray[np.float64]:
    """Convert ratings DataFrame to a user-item matrix."""
    num_users = ratings["user_id"].max() + 1
    num_items = ratings["item_id"].max() + 1

    user_item_matrix = np.full((num_users, num_items), fill_value, dtype=np.float64)
    user_item_matrix[ratings["user_id"], ratings["item_id"]] = ratings["rating"].values.astype(np.float64)

    return user_item_matrix


In [ ]:
ratings, users, movies = load_movielens()
density = len(ratings) / (len(users) * len(movies))
print(f"notas={len(ratings):,}; usuários={len(users):,}; filmes={len(movies):,}; densidade={density:.3f}")

## Formulação do problema

Uma suposição comum em sistemas de recomendação é que a preferência de um usuário por um item pode ser representada como uma função sobre um espaço latente compartilhado entre usuários e itens.
Ou seja, considerando que cada usuário e cada item podem ser representados como vetores em um espaço latente, a nota de um usuário para um item pode ser modelada como uma função que mede a compatibilidade entre os vetores do usuário e do item.

Utilizaremos uma formulação geral para este problema, que pode acomodar diferentes modelos e suposições sobre como os vetores latentes são gerados e como a função de compatibilidade é definida.
Considere um conjunto de usuários $U$ e um conjunto de itens $I$, e seja $R$ a matriz de notas, a nota do usuário $u$ para o item $i$, $R_{ui}$ é modelada como:

$$
R_{ui} = \mu + b_u + b_i + f(\mathbf{z}_u, \mathbf{w}_i) + \epsilon_{ui}
$$
onde $\mu$ é a média global das notas, $b_u$ e $b_i$ são os *biases* do usuário e do item, respectivamente, $\mathbf{z}_u$ e $\mathbf{w}_i$ são os vetores latentes do usuário e do item, $f$ é a função de compatibilidade entre os vetores latentes e $\epsilon_{ui}$ é o erro de modelagem para aquele par usuário-item.

Um desafio importante é que a matriz de notas $R$ é muito esparsa, ou seja, grande parte dos pares usuário-item não possuem notas observadas. Isso significa que aprender a estrutura latente compartilhada pode ser difícil sem fazer suposições adicionais sobre como os vetores latentes são gerados e como a função de compatibilidade é definida.

Vamos explorar como diferentes suposições afetam o desempenho do modelo e o que podemos aprender sobre a estrutura latente dos dados através da inspeção do espaço latente aprendido.

> **Importante:** Os modelos têm uma abordagem transdutiva, ou seja, eles aprendem os vetores latentes apenas para os usuários e itens presentes no conjunto de treinamento.
As features de um usuário não são as notas que ele deu, mas sim apenas um identificador único, e o mesmo vale para os itens.
Isso permite que o modelo possua parâmetros específicos para cada usuário, mas também significa que ele não pode generalizar para novos usuários ou itens sem re-treinamento.

## Diretrizes para o trabalho

Para cada experimento, sigam as seguintes diretrizes:

> 1. Escrevam a hipótese e a previsão **antes** de rodar cada experimento.
> 2. Resultados negativos ou inconclusivos valem, se vierem com **análise crítica**.
> 3. Todos os modelos, inclusive os mais simples, passam pelo mesmo protocolo (a seguir).
> 4. Separem o que um resultado diz sobre a previsão do que ele diz sobre a representação latente.
> 5. Citem fontes sempre que necessário, seja para motivar uma escolha, ou para justificar uma decisão.

A nota não depende da métrica obtida pelo modelo, mas da qualidade das hipóteses, aplicação correta do protocolo e da análise das evidências a favor, contra ou insuficientes.

### Protocolo experimental

Os experimentos devem ser controlados, com uma variável de interesse por vez e com o mesmo protocolo:

**Split dos dados.** O conjunto de dados deve ser dividido em treino e teste, com 80% das notas para treino e 20% para teste.
A divisão deve ser feita de forma estratificada por usuário, garantindo que cada usuário tenha pelo menos uma nota em cada conjunto, e todo filme deve ter pelo menos uma nota no conjunto de treino.

**Métricas.** RMSE (principal) e MAE, com previsões truncadas em $[1,5]$, e NLL quando o modelo define uma verossimilhança comparável.

**Seleção de hiperparâmetros.** O conjunto de teste é usado uma única vez, para reportar o desempenho final.
Os hiperparâmetros (dimensão latente, regularização, variância do ruído, número de épocas e assim por diante) são escolhidos por *cross-validation* com $K$ folds dentro do conjunto de treino, respeitando as mesmas restrições da divisão de treino e teste.
Depois da escolha, o modelo é treinado com todo o treino e avaliado no teste.

**Incerteza.** Compare resultados com **intervalos de confiança** sobre as métricas.
Vocês podem usar *cross-validation*, reportando a média e o desvio padrão das métricas entre os folds, que refletem a variação causada pelos dados de treino e pela inicialização, ou *bootstrapping*, amostrando com reposição o conjunto de teste e reportando a média e o intervalo de percentis de 95%, que reflete a variação da amostra de teste.

In [ ]:
def split_ratings(
    ratings: pd.DataFrame,
    test_frac: float = 0.2,
    seed: int = 0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = (
        ratings.sample(frac=1, random_state=seed)
               .sort_values("user_id", kind="stable")
               .reset_index(drop=True)
    )

    grouped = df.groupby("user_id")
    pos = (
        grouped.cumcount() /
        grouped["user_id"].transform("count")
    ).to_numpy()

    splits = np.zeros(len(df), dtype=np.int8)
    splits[pos >= (1 - test_frac)] = 1

    # Trocar a primeira aparição de cada filme para o conjunto de treino
    first_appearance = df.groupby("item_id").head(1).index
    splits[first_appearance] = 0

    train_df = df.iloc[splits == 0]
    test_df = df.iloc[splits == 1]

    return train_df, test_df

In [ ]:
from typing import Protocol

from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
)

# Funções auxiliares

def clip_ratings(prediction: NDArray[np.float64]) -> NDArray[np.float64]:
    return np.clip(np.asarray(prediction, dtype=np.float64), 1.0, 5.0)


class Model(Protocol):
    def fit(self, train_df: DataFrame) -> None: ...

    def predict(
        self, user_id: NDArray[np.intp], item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]: ...


class ProbabilisticModel(Model, Protocol):
    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
    ) -> NDArray[np.float64]: ...


def evaluate_model(
    model: Model | ProbabilisticModel, **dfs: DataFrame
) -> dict[str, float]:
    metrics: dict[str, float] = {}
    
    for name, df in dfs.items():
        users = df["user_id"].to_numpy(dtype=np.intp)
        items = df["item_id"].to_numpy(dtype=np.intp)
        ratings = df["rating"].to_numpy(dtype=np.float64)

        preds = model.predict(users, items)
        preds_clipped = clip_ratings(preds)

        metrics |= {
            f"{name}_rmse": root_mean_squared_error(ratings, preds_clipped),
            f"{name}_mae": mean_absolute_error(ratings, preds_clipped),
            # Vocês podem adicionar mais métricas aqui
        }

        if hasattr(model, "predict_log_proba"):
            log_proba = model.predict_log_proba(users, items, ratings)
            metrics[f"{name}_nll"] = float(-np.mean(log_proba))

    return metrics



In [ ]:
train_df, test_df = split_ratings(ratings, seed=SEED)

In [ ]:
# Explorem os dados, plotem gráficos, calculem estatísticas, para entender melhor o problema,
# os dados disponíveis, a distribuição das notas, a distribuição de usuários e itens, e assim por diante.

## Etapa 1: Baselines

O primeiro passo para entender o problema é estabelecer uma baseline, um modelo simples que sirva como referência da performance que podemos esperar de um modelo mais sofisticado.
Podemos simplificar a formulação geral do problema, considerando apenas a média global das notas e os *biases* do usuário e do item, sem considerar os vetores latentes:

$$R_{ui} = \mu + b_u + b_i + \epsilon_{ui}$$

Como o modelo remove a interação entre usuários e itens, ele não é capaz de capturar a estrutura latente compartilhada entre eles, mas ainda assim pode fornecer uma boa referência para avaliar o desempenho de modelos mais complexos.
[Koren, Y. et. al.](https://ieeexplore.ieee.org/document/5197422) discute este exato modelo como uma aproximação de primeira ordem para o problema, que pode ser melhorada com a adição de termos de segunda ordem, como os vetores latentes introduzidos na formulação geral.
Recomenda-se a leitura do artigo para entender melhor a motivação e interpretação do problema e as soluções propostas.

Devido à linealidade do modelo,podemos aprender os *biases* do usuário e do item de forma eficiente, mesmo com a matriz de notas esparsa, através de regressão linear regularizada.

> **Tarefa 1.1**
> Treine o modelo de baseline e avalie seu desempenho, reportando RMSE e MAE no conjunto de teste.
> Lembre-se de usar o mesmo protocolo experimental para todos os modelos para encontrar o melhor conjunto de hiperparâmetros.
> Após treinar o modelo, analise os *biases* aprendidos e discuta o que eles podem nos dizer sobre a preferência dos usuários e a popularidade dos itens.

In [ ]:
class BaselineModel:
    def __init__(
        self,
        user_reg: float = 10.0,
        item_reg: float = 10.0,
        n_epochs: int = 20,
    ) -> None:
        self.user_reg = user_reg
        self.item_reg = item_reg
        self.n_epochs = n_epochs

    def fit(self, train_df: pd.DataFrame) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.intp)
        items = train_df["item_id"].to_numpy(dtype=np.intp)

        self.global_mean = float(ratings.mean())
        self.user_bias = np.zeros(int(users.max()) + 1, dtype=np.float64)
        self.item_bias = np.zeros(int(items.max()) + 1, dtype=np.float64)

        n_users = len(self.user_bias)
        n_items = len(self.item_bias)

        user_counts = np.bincount(users, minlength=n_users)
        item_counts = np.bincount(items, minlength=n_items)

        for _ in range(self.n_epochs):
            # Os vieses de usuário e item são atualizados alternadamente, mantendo o outro fixo.
            # cada um resolve \min_b (sum_{(u,i) in R} (r_{ui} - mu - b_u - b_i)^2 + lambda * b^2)
            # Que é equivalente à agrupar por usuário e item, e então resolver a equação normal para cada um.

            user_residuals = ratings - self.global_mean - self.item_bias[items]
            self.user_bias = np.bincount(
                users, weights=user_residuals, minlength=n_users
            ) / (user_counts + self.user_reg)

            item_residuals = ratings - self.global_mean - self.user_bias[users]
            self.item_bias = np.bincount(
                items, weights=item_residuals, minlength=n_items
            ) / (item_counts + self.item_reg)

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.global_mean
            + self.user_bias[user_id]
            + self.item_bias[item_id]
        )

In [ ]:
# Faça o treinamento e a avaliação do modelo baseline aqui

**Discussão:**
*Escreva suas conclusões aqui*

---
### Introduzindo vetores latentes

A baseline linear captura diferenças sistemáticas entre usuários e itens, mas não explica por que determinados grupos de usuários atribuem notas semelhantes a determinados grupos de itens.
Para investigar essa estrutura, podemos olhar para a matriz de notas como um todo e perguntar se sua variabilidade pode ser explicada por um número reduzido de dimensões.
Essa perspectiva permite explorar interações globais nos dados e construir uma representação mais compacta das avaliações, codificando usuários e itens em um espaço latente compartilhado.

Uma maneira de investigar essa hipótese é completar a matriz de notas e aplicar Análise de Componentes Principais (PCA, *Principal Component Analysis*).
Para isso, substituímos as notas ausentes pela média do respectivo item, calculada apenas com as avaliações observadas no conjunto de treinamento.
Em seguida, o PCA identifica as principais direções de variação da matriz completada, permitindo representar cada usuário e cada item como vetores em um espaço de baixa dimensionalidade.

Seja $\tilde{R}$ a matriz de notas em que cada nota ausente foi substituída por um valor de imputação, por exemplo, a média do item.
O PCA com $k$ componentes aproxima $\tilde{R}$ por uma matriz de posto $k$:

$$\hat{R}_{ui} = \bar{R}_i + \mathbf{z}_u^\top \mathbf{w}_i$$

onde $\bar{R}_i$ é a média da coluna $i$ de $\tilde{R}$, $\mathbf{z}_u$ e $\mathbf{w}_i$ são as coordenadas do usuário $u$ e do item $i$ nos $k$ primeiros componentes principais.
Sob a perspectiva do PCA, um usuário é codificado como um vetor de pesos que indica a importância de cada componente principal para aquele usuário, que então é decodificado em uma nota para cada item, através da combinação linear com a transformação dos itens no mesmo espaço latente.

> **Tarefa 1.2**
Treine o modelo de PCA com imputação de notas ausentes, utilizando a média de cada item para preencher os valores faltantes. Utilize o protocolo experimental para selecionar o número de componentes principais $k$ e então reporte RMSE e MAE no conjunto de teste para o valor de $k$ selecionado.
>
> Analise e discuta também como o desempenho varia com o número de componentes, discutindo se o aumento da dimensionalidade melhora as previsões, produz ganhos decrescentes ou degrada o desempenho.


In [ ]:
class ImputedPCA:
    def __init__(self, n_components: int = 10) -> None:
        self.n_components = n_components
        self.user_embeddings: NDArray[np.float64]
        self.item_embeddings: NDArray[np.float64]
        self.item_means: NDArray[np.float64]
        self.pca = PCA(n_components=n_components)

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)

        # Imputação de valores ausentes com a média do item
        observed = ~np.isnan(X)
        global_mean = float(X[observed].mean())
        counts = observed.sum(axis=0)
        sums = np.where(observed, X, 0.0).sum(axis=0)

        self.item_means = np.divide(
            sums,
            counts,
            out=np.full(X.shape[1], global_mean),
            where=counts > 0,
        )

        rows, cols = np.where(~observed)
        X[rows, cols] = self.item_means[cols]

        user_scores = self.pca.fit_transform(X)
        self.user_embeddings = user_scores
        self.item_embeddings = self.pca.components_.T

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.pca.mean_[item_id]
            + np.sum(
                self.user_embeddings[user_id]
                * self.item_embeddings[item_id],
                axis=1,
            )
        )

In [ ]:
# Faça o treinamento e a avaliação do PCA imputado aqui

**Discussão:**
*Escreva suas conclusões aqui*

---

### Fatoração de matrizes (*Matrix Factorization*)

A principal limitação do modelo anterior é que ele depende de notas imputadas, que não correspondem a avaliações reais e podem introduzir vieses no aprendizado dos vetores latentes.
Para superar essa limitação, devemos aprender os vetores latentes diretamente a partir das notas observadas, sem precisar completar previamente a matriz.

A fatoração de matrizes parte da hipótese de que a matriz de notas pode ser aproximada pelo produto de duas matrizes de menor dimensão, ou seja

$$\hat R = Z W^\top,$$
em que $Z\in\mathbb R^{|U|\times k}$ e $W\in\mathbb R^{|I|\times k}$ contêm os vetores latentes dos usuários e dos itens, respectivamente.
Dessa forma, a nota prevista para um par usuário–item é dada pelo produto interno entre suas representações:

$$\hat R_{ui}=\mathbf z_u^\top\mathbf w_i.$$

Entretanto, como vimos na baseline, parte da variação das notas pode ser explicada por diferenças sistemáticas entre usuários e itens, independentemente de suas interações (por exemplo, alguns filmes são mais populares que outros).
Para considerar esses efeitos, incorporamos a média global e os *biases* de usuário e item ao modelo, obtendo a formulação geral com $f(\mathbf{z}_u, \mathbf{w}_i) = \mathbf z_u^\top\mathbf w_i$:

$$\hat R_{ui}=\mu+b_u+b_i+\mathbf z_u^\top\mathbf w_i.$$

Para aprender os parâmetros, minimizamos o erro quadrático entre as notas observadas e as respectivas previsões, considerando apenas os pares usuário–item avaliados no conjunto de treinamento.
Introduzimos também regularização $L_2$, que penaliza valores elevados dos *biases* e das normas dos vetores latentes:

\begin{aligned}
\min_{\mu,b,Z_U,Z_I}\quad
&\sum_{(u,i)\in\Omega}
\left(r_{ui}-\mu-b_u-b_i-\mathbf z_u^\top\mathbf w_i\right)^2\\
&+\lambda_b\left(\sum_u b_u^2+\sum_i b_i^2\right)\\
&+\lambda_z\left(\sum_u\|\mathbf z_u\|_2^2+
\sum_i\|\mathbf w_i\|_2^2\right),
\end{aligned}

em que $\Omega$ é o conjunto de avaliações observadas no treinamento, enquanto $\lambda_b$ e $\lambda_z$ controlam a intensidade da regularização dos *biases* e dos fatores latentes, respectivamente.
Como o problema é não-convexo, sem solução fechada, usamos descida no gradiente para atualizar os parâmetros, iterativamente com base em pequenas amostras do conjunto de treinamento.

> **Tarefa 1.3**
Treine um modelo de fatoração de matrizes utilizando regularização $L_2$.
Selecione o número de dimensões latentes $k$ e os coeficientes de regularização com base no RMSE.
> 
> Reporte RMSE e MAE no conjunto de teste para a configuração selecionada.
Analise como o desempenho varia com $k$ e com a regularização.
Por fim, compare a fatoração com *biases* e sem *biases*, mantendo o mesmo protocolo experimental, para investigar a contribuição dos efeitos aditivos e das interações latentes para a qualidade das previsões.
>
> Siga o mesmo protocolo experimental utilizado para os modelos anteriores.


In [ ]:
class MatrixFactorization:
    user_embeddings: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    global_mean: float
    user_bias: NDArray[np.float64]
    item_bias: NDArray[np.float64]

    def __init__(
        self,
        n_factors: int = 10,
        n_epochs: int = 20,
        learning_rate: float = 0.01,
        reg: float = 0.1,
        bias_reg: float | None = None,
        use_biases: bool = True,
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.learning_rate = learning_rate
        self.reg = reg
        self.bias_reg = reg if bias_reg is None else bias_reg
        self.use_biases = use_biases

    def fit(self, train_df: pd.DataFrame) -> None:
        ratings = train_df["rating"].to_numpy(dtype=np.float64)
        users = train_df["user_id"].to_numpy(dtype=np.intp)
        items = train_df["item_id"].to_numpy(dtype=np.intp)

        self.global_mean = float(ratings.mean())
        n_users = int(users.max()) + 1
        n_items = int(items.max()) + 1

        self.user_embeddings = np.random.normal(scale=0.1, size=(n_users, self.n_factors))
        self.item_embeddings = np.random.normal(scale=0.1, size=(n_items, self.n_factors))
        self.user_bias = np.zeros(n_users, dtype=np.float64)
        self.item_bias = np.zeros(n_items, dtype=np.float64)

        for _ in range(self.n_epochs):
            for n in np.random.permutation(len(ratings)):
                u, i, r = users[n], items[n], ratings[n]
                pred = (
                    self.global_mean
                    + self.user_bias[u]
                    + self.item_bias[i]
                    + np.dot(self.user_embeddings[u], self.item_embeddings[i])
                )
                err = r - pred

                # Atualização dos vieses
                if self.use_biases:
                    self.user_bias[u] += self.learning_rate * (err - self.bias_reg * self.user_bias[u])
                    self.item_bias[i] += self.learning_rate * (err - self.bias_reg * self.item_bias[i])

                user_emb_old = self.user_embeddings[u].copy()
                item_emb_old = self.item_embeddings[i].copy()

                self.user_embeddings[u] += self.learning_rate * (err * item_emb_old - self.reg * user_emb_old)
                self.item_embeddings[i] += self.learning_rate * (err * user_emb_old - self.reg * item_emb_old)

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return (
            self.global_mean
            + self.user_bias[user_id]
            + self.item_bias[item_id]
            + np.sum(
                self.user_embeddings[user_id] * self.item_embeddings[item_id],
                axis=1,
            )
        )


In [ ]:
# Faça o treinamento e a avaliação do modelo de fatoração de matrizes aqui

**Discussão:**
*Escreva suas conclusões aqui*

---
## Etapa 2: Suposições sobre a estrutura latente

Na etapa anterior, investigamos modelos que estimam notas a partir de vieses aditivos e representações latentes de usuários e itens.
A fatoração de matrizes procura representações que aproximem as avaliações observadas, mas não define, por si só, uma distribuição probabilística completa para os dados.

Nesta etapa, exploraremos uma abordagem probabilística baseada no PCA Probabilístico (*Probabilistic Principal Component Analysis*, PPCA), adaptada ao problema de recomendação.
Essa formulação descreve as avaliações por meio de variáveis latentes e ruído gaussiano, permitindo explicitar a variância do ruído, estimar a verossimilhança dos dados e investigar como as suposições probabilísticas afetam as previsões.

Ao contrário das implementações clássicas de PPCA e VAE, em que o encoder é treinado para mapear as observações para a distribuição posterior, aqui, como os usuários do conjunto de dados são conhecidos durante o treinamento, podemos estimar um embedding para cada um deles e armazenar essas representações após o ajuste do modelo.
Isso é equivalente a um *encoder* que mapeia cada usuário para uma distribuição posterior com o máximo de liberdade, sem lidar com features de usuário, vieses indutivos ou generalização para novos usuários.


### PCA Probabilístico

O PCA clássico identifica um subespaço linear de dimensão reduzida que representa a estrutura de variância dos dados. O PPCA estende essa ideia ao assumir que as observações são geradas por transformações lineares de variáveis latentes gaussianas, acrescidas de ruído gaussiano.

No problema de recomendação, podemos representar a matriz de avaliações por $R\in\mathbb R^{n\times m}$, em que $n$ é o número de usuários e $m$ é o número de filmes. Cada usuário corresponde a uma observação, e cada filme corresponde a uma dimensão observada. Como os usuários do conjunto de dados são conhecidos durante o treinamento, podemos estimar um embedding para cada um deles e armazenar essas representações após o ajuste do modelo.

Seja $\mathbf z_u\in\mathbb R^k$ a variável latente do usuário $u$, e seja $\mathbf w_i\in\mathbb R^k$ o vetor de cargas associado ao filme $i$. O modelo assume que

$$
\mathbf z_u\sim\mathcal N(\mathbf 0,I_k)
$$

e que a avaliação do usuário $u$ para o filme $i$ segue

$$
R_{ui}\mid\mathbf z_u
\sim
\mathcal N\left(
\mu_i+\mathbf z_u^\top\mathbf w_i,
\sigma^2
\right),
$$

em que $\mu_i$ é a média associada ao filme $i$, $k$ é a dimensão latente e $\sigma^2$ é a variância do ruído, considerada fixa durante o treinamento.
A média condicional da avaliação é, portanto,

$$
\mathbb E[R_{ui}\mid\mathbf z_u]
=
\mu_i+\mathbf z_u^\top\mathbf w_i.
$$

Essa formulação representa as avaliações por meio de uma estrutura linear de baixa dimensão, sem introduzir vieses aditivos específicos de usuários.
A diferença fundamental em relação à fatoração de matrizes determinística está na distribuição probabilística atribuída às avaliações e às variáveis latentes.

O treinamento pode ser realizado pelo algoritmo EM (*Expectation-Maximization*).
Seja $\Omega_u$ o conjunto de filmes avaliados pelo usuário $u$ e $\Omega_i$ o conjunto de usuários que avaliaram o filme $i$.
Na etapa E, como o modelo é linear e gaussiano, a posterior de cada usuário, condicionada às avaliações observadas e aos parâmetros atuais, é gaussiana, $p(\mathbf z_u\mid R_u)=\mathcal N(\mathbf m_u, S_u)$, com

$$
S_u=\left( I_k + \frac{1}{\sigma^2} \sum_{i\in\Omega_u} \mathbf w_i\mathbf w_i^\top\right)^{-1},
\qquad
\mathbf m_u=\frac{1}{\sigma^2} \, S_u\sum_{i\in\Omega_u}\mathbf w_i\,(r_{ui}-\mu_i).
$$

Na etapa M, atualizamos as médias dos filmes e os vetores de cargas para maximizar a verossimilhança esperada.
Definindo $\boldsymbol\theta_i=(\mu_i,\mathbf w_i)$ e $\boldsymbol\phi_u=(1,\mathbf z_u)$, cada filme resolve um sistema de mínimos quadrados em que as posteriores entram pelos momentos $\mathbb E[\boldsymbol\phi_u]=(1,\mathbf m_u)$ e $\mathbb E[\mathbf z_u\mathbf z_u^\top]=S_u+\mathbf m_u\mathbf m_u^\top$:

$$
\boldsymbol\theta_i=\left(\sum_{u\in\Omega_i}\mathbb E[\boldsymbol\phi_u\boldsymbol\phi_u^\top]\right)^{-1}\sum_{u\in\Omega_i}r_{ui}\,\mathbb E[\boldsymbol\phi_u].
$$

Como a matriz de avaliações é esparsa, cada usuário contribui para a estimação apenas por meio dos filmes que avaliou. As avaliações ausentes são tratadas como dados não observados, e não como notas iguais a zero. Após o treinamento, as médias dos embeddings posteriores podem ser armazenadas como representações dos usuários, permitindo realizar previsões diretamente para os pares usuário–filme.

Para uma avaliação observada $r_{ui}$, a densidade condicional é

$$
p(r_{ui}\mid\mathbf z_u)
=
\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\left(
-\frac{
(r_{ui}-\mu_i-\mathbf w_i^\top\mathbf z_u)^2
}{2\sigma^2}
\right).
$$

A log-verossimilhança conjunta das avaliações observadas, condicionada às variáveis latentes, é obtida somando as contribuições dos pares usuário–filme observados. O prior gaussiano das variáveis latentes também participa do modelo probabilístico, favorecendo representações compatíveis com a distribuição $\mathcal N(\mathbf 0,I_k)$.

Para prever a avaliação de um par usuário–filme, utilizamos a média condicional calculada com o embedding posterior estimado:

$$
\hat R_{ui}
=
\mu_i+\mathbf w_i^\top\mathbb E[\mathbf z_u\mid R_u],
$$

em que $R_u$ representa as avaliações observadas do usuário $u$. Essa previsão corresponde à média da distribuição preditiva quando os parâmetros do modelo são considerados fixos e a incerteza sobre a variável latente é integrada.

A variância $\sigma^2$ controla o ruído residual assumido pelo modelo. Valores menores atribuem maior concentração às avaliações em torno da média condicional; valores maiores representam maior dispersão. Como a variância é fixada durante o treinamento nesta implementação, ela deve ser selecionada como hiperparâmetro.

> **Tarefa 2.1**
Treine o modelo PPCA para a matriz esparsa de avaliações e selecione o número de dimensões latentes $k$ e a variância do ruído $\sigma^2$ usando o protocolo experimental.
Compare o RMSE com os modelos da etapa anterior e investigue se a configuração que minimiza o RMSE também produz a melhor log-verossimilhança negativa média (*Negative Log-Likelihood*, NLL).
>
> Discuta como a variância do ruído afeta a distribuição das avaliações, como o prior gaussiano influencia as representações latentes e quais são as implicações de estimar embeddings específicos para os usuários conhecidos durante o treinamento.

In [ ]:
class PPCA:
    user_embeddings: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    item_means: NDArray[np.float64]
    user_covariances: NDArray[np.float64]

    def __init__(
        self, n_factors: int = 10, n_epochs: int = 100, sigma2: float = 1.0
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.sigma2 = sigma2

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)
        weights = observed.astype(np.float64)

        counts = observed.sum(axis=0)
        global_mean = values.sum() / observed.sum()
        self.item_means = np.divide(
            values.sum(axis=0),
            counts,
            out=np.full(X.shape[1], global_mean),
            where=counts > 0,
        )

        n_users, n_items = X.shape
        k = self.n_factors
        sigma2 = self.sigma2
        eye = np.eye(k)

        self.item_embeddings = np.random.normal(
            scale=0.1, size=(n_items, k)
        )

        item_outer = (
            self.item_embeddings[:, :, None]
            * self.item_embeddings[:, None, :]
        )

        for _ in range(self.n_epochs):
            centered = np.where(observed, X - self.item_means, 0.0)

            precision = eye + np.einsum("ui,ijk->ujk", weights, item_outer) / sigma2

            self.user_covariances = np.linalg.inv(precision)

            rhs = centered @ self.item_embeddings / sigma2
            self.user_embeddings = (
                self.user_covariances @ rhs[:, :, None]
            )[:, :, 0]

            user_means = np.column_stack(
                (np.ones(n_users), self.user_embeddings)
            )

            user_second = np.zeros((n_users, k + 1, k + 1))
            user_second[:, 0, 0] = 1.0
            user_second[:, 0, 1:] = self.user_embeddings
            user_second[:, 1:, 0] = self.user_embeddings
            user_second[:, 1:, 1:] = (
                self.user_covariances
                + self.user_embeddings[:, :, None]
                * self.user_embeddings[:, None, :]
            )

            A = np.einsum("ui,uab->iab", weights, user_second)

            b = values.T @ user_means
            theta = np.linalg.solve(A, b[:, :, None])[:, :, 0]

            self.item_means = theta[:, 0]
            self.item_embeddings = theta[:, 1:]

            item_outer = (
                self.item_embeddings[:, :, None]
                * self.item_embeddings[:, None, :]
            )

        centered = np.where(observed, X - self.item_means, 0.0)

        precision = eye + np.einsum("ui,ijk->ujk", weights, item_outer) / sigma2

        self.user_covariances = np.linalg.inv(precision)

        rhs = centered @ self.item_embeddings / sigma2
        self.user_embeddings = (
            self.user_covariances @ rhs[:, :, None]
        )[:, :, 0]

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        return self.item_means[item_id] + np.sum(
            self.user_embeddings[user_id] * self.item_embeddings[item_id],
            axis=1,
        )

    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
    ) -> NDArray[np.float64]:
        user_embeddings = self.user_embeddings[user_id]
        item_embeddings = self.item_embeddings[item_id]
        covariances = self.user_covariances[user_id]

        means = self.item_means[item_id] + np.sum(
            user_embeddings * item_embeddings, axis=1
        )

        variances = self.sigma2 + np.einsum(
            "ni,nij,nj->n",
            item_embeddings,
            covariances,
            item_embeddings,
        )

        return -0.5 * (
            np.log(2 * np.pi * variances)
            + (ratings - means) ** 2 / variances
        )

In [ ]:
# Faça o treinamento e a avaliação do PPCA aqui

**Discussão:**
*Escreva suas conclusões aqui*

### Autoencoder variacional (VAE)

O PPCA modela as avaliações por meio de uma transformação linear de variáveis latentes gaussianas. O VAE estende essa abordagem ao permitir um decodificador não linear e aprender uma distribuição latente específica para cada usuário.

Seja $k$ a dimensão latente.
O codificador é representado por uma matriz de embeddings de usuários, que fornece a média latente $\mathbf m_u$ e uma matriz de parâmetros que fornece a variância diagonal $\mathbf s_u^2$ de cada usuário.
Assim, a distribuição latente aproximada é

$$
\mathbf z_u
\sim
\mathcal N\left(
\mathbf m_u,
\operatorname{diag}(\mathbf s_u^2)
\right),
$$
que pode ser amostrada por meio do truque de reparametrização:

$$
\mathbf z_u
=
\mathbf m_u+\mathbf s_u\odot\boldsymbol\epsilon,
\qquad
\boldsymbol\epsilon\sim\mathcal N(\mathbf 0,I_k).
$$

O decodificador transforma essa amostra em previsões para todos os filmes através de uma função $g_\theta:\mathbb R^k\to\mathbb R^m$ parametrizada por $\theta$:

$$
\hat{\mathbf R}_u = g_\theta(\mathbf z_u).
$$

Assumimos o seguinte modelo generativo:

$$
R_{ui}\mid\mathbf z_u
\sim
\mathcal N\left(g_\theta(\mathbf z_u)_i,\sigma^2\right),
$$

em que $\sigma^2$ é a variância do ruído das avaliações.
Com um decodificador linear, $g_\theta(\mathbf z)=\boldsymbol\mu+W\mathbf z$, a média condicional das avaliações é uma função linear da variável latente, como no PPCA.
Um decodificador não linear permite representar relações mais complexas entre as dimensões latentes e as avaliações.

O treinamento maximiza o limite inferior da log-verossimilhança (*Evidence Lower Bound*, ELBO), somado sobre os usuários:

$$
\mathcal L_u
=
\mathbb E_{q_\phi(\mathbf z_u)}
\left[
-\frac{1}{2\sigma^2}
\sum_{i\in\Omega_u}
(r_{ui}-g_\theta(\mathbf z_u)_i)^2
\right] -\beta\,\mathrm{KL}\left(
q_\phi(\mathbf z_u)
\,\|\,\mathcal N(\mathbf 0,I_k)
\right),
$$
em que $\Omega_u$ é o conjunto de filmes avaliados pelo usuário.
Foram omitidas constantes que não dependem dos parâmetros do modelo.
Com $\beta=1$, obtemos o ELBO padrão, a menos dessas constantes, variar $\beta$ altera o peso relativo da regularização latente.

O primeiro termo incentiva o modelo a atribuir alta probabilidade às notas observadas, o segundo aproxima a distribuição latente de cada usuário do prior gaussiano padrão, regularizando as representações aprendidas.
A esperança é estimada por amostragem da distribuição latente, permitindo propagar gradientes pelos parâmetros do codificador e do decodificador.

Na predição, podemos usar a média latente $\mathbf m_u$ como embedding do usuário e calcular

$$
\hat R_{ui} = g_\theta(\mathbf m_u)_i.
$$

Essa escolha fornece uma previsão pontual determinística. A distribuição preditiva completa, por sua vez, também considera a variabilidade latente e o ruído das avaliações.

Diferentemente do PPCA, que calcula a posterior de cada usuário analiticamente a partir das avaliações observadas, o VAE aprende por otimização uma distribuição latente específica para cada usuário, representada por uma média e uma variância. Essa formulação é transdutiva: as representações são aprendidas para os usuários presentes no treinamento e não se estendem automaticamente a usuários novos.

> **Tarefa 2.2**
Treine um VAE com decodificador linear e compare seu desempenho com o PPCA usando o mesmo número de dimensões latentes $k$ e a mesma variância do ruído $\sigma^2$. Compare RMSE, MAE e NLL no conjunto de teste.
>
> Em seguida, substitua o decodificador linear por um não linear e investigue se a flexibilidade adicional melhora as previsões. Escolha um hiperparâmetros para variar isoladamente, $\beta$, $\sigma^2$ ou $k$, e aplique o protocolo experimental.
Verifique a convergência pela curva de treinamento armazenada em `history` antes de comparar os modelos.

Atenção, com um decoder não-linear, os itens 

In [ ]:
import torch
from torch import nn


class VAE:
    user_embeddings: NDArray[np.float64]
    user_variances: NDArray[np.float64]
    item_embeddings: NDArray[np.float64]
    item_means: NDArray[np.float64]

    def __init__(
        self,
        n_factors: int = 10,
        n_epochs: int = 300,
        sigma2: float = 1.0,
        beta: float = 1.0,
        hidden_dim: int | None = None,
        learning_rate: float = 1e-2,
    ) -> None:
        self.n_factors = n_factors
        self.n_epochs = n_epochs
        self.sigma2 = sigma2
        self.beta = beta
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        self.history: list[float] = []

    def fit(self, train_df: pd.DataFrame) -> None:
        X = ratings_to_matrix(train_df)
        observed = ~np.isnan(X)
        values = np.where(observed, X, 0.0)

        X = torch.tensor(values, dtype=torch.float32)
        mask = torch.tensor(observed, dtype=torch.float32)

        n_users, n_items = X.shape
        k = self.n_factors

        self.user_means = nn.Parameter(torch.randn(n_users, k) * 0.01)
        self.user_logvars = nn.Parameter(torch.zeros(n_users, k))

        if self.hidden_dim is None:
            self.decoder = nn.Linear(k, n_items)

        else:
            self.decoder = nn.Sequential(
                nn.Linear(k, self.hidden_dim),
                nn.Tanh(),
                nn.Linear(self.hidden_dim, n_items),
            )

        # O viés do decodificador começa nas médias dos filmes, e o treino modela o desvio em relação a elas
        counts = np.maximum(observed.sum(axis=0), 1)
        item_means = np.where(observed.any(axis=0), values.sum(axis=0) / counts, values.sum() / observed.sum())
        output_layer = self.decoder if self.hidden_dim is None else self.decoder[-1]
        with torch.no_grad():
            output_layer.bias.copy_(torch.tensor(item_means, dtype=torch.float32))

        parameters = [
            self.user_means,
            self.user_logvars,
            *self.decoder.parameters(),
        ]
        optimizer = torch.optim.Adam(parameters, lr=self.learning_rate)

        self.history = []

        for _ in range(self.n_epochs):
            optimizer.zero_grad()

            std = torch.exp(0.5 * self.user_logvars)
            epsilon = torch.randn_like(std)
            z = self.user_means + std * epsilon

            predictions = self.decoder(z)

            log_likelihood = -0.5 * (
                mask
                * (
                    np.log(2 * np.pi * self.sigma2)
                    + (X - predictions) ** 2 / self.sigma2
                )
            ).sum()

            kl = -0.5 * (
                1
                + self.user_logvars
                - self.user_means**2
                - self.user_logvars.exp()
            ).sum()

            elbo = log_likelihood - self.beta * kl
            loss = -elbo / n_users

            loss.backward()
            optimizer.step()

            self.history.append((-loss.detach()).item())

        with torch.no_grad():
            self.user_embeddings = (
                self.user_means.detach().cpu().numpy().astype(np.float64)
            )
            self.user_variances = (
                self.user_logvars.exp()
                .detach()
                .cpu()
                .numpy()
                .astype(np.float64)
            )

            if self.hidden_dim is None:
                self.item_embeddings = (
                    self.decoder.weight.detach()
                    .cpu().numpy().astype(np.float64)
                )
                self.item_means = (
                    self.decoder.bias.detach()
                    .cpu().numpy().astype(np.float64)
                )

    def _decode_at(self, z: torch.Tensor, items: torch.Tensor) -> torch.Tensor:
        """Decode only the entries (z[..., n, :], items[n]), without computing every movie."""
        if self.hidden_dim is None:
            return (z * self.decoder.weight[items]).sum(-1) + self.decoder.bias[items]
        first, _, last = self.decoder
        return (torch.tanh(first(z)) * last.weight[items]).sum(-1) + last.bias[items]

    def predict(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
    ) -> NDArray[np.float64]:
        users = torch.as_tensor(user_id, dtype=torch.long)
        items = torch.as_tensor(item_id, dtype=torch.long)

        with torch.no_grad():
            z = torch.as_tensor(self.user_embeddings, dtype=torch.float32)[users]
            result = self._decode_at(z, items)

        return result.cpu().numpy().astype(np.float64)

    def predict_log_proba(
        self,
        user_id: NDArray[np.intp],
        item_id: NDArray[np.intp],
        ratings: NDArray[np.float64],
        n_samples: int = 100,
        chunk: int = 2048,
    ) -> NDArray[np.float64]:
        """Log of the Monte Carlo predictive density, computed in chunks of rating pairs."""
        users = torch.as_tensor(user_id, dtype=torch.long)
        items = torch.as_tensor(item_id, dtype=torch.long)
        ratings_t = torch.as_tensor(ratings, dtype=torch.float32)
        user_means = torch.as_tensor(self.user_embeddings, dtype=torch.float32)
        user_stds = torch.as_tensor(self.user_variances, dtype=torch.float32).sqrt()

        result = []
        with torch.no_grad():
            for start in range(0, len(users), chunk):
                u = users[start : start + chunk]
                means, stds = user_means[u], user_stds[u]
                z = means + stds * torch.randn(n_samples, *means.shape)
                predictions = self._decode_at(z, items[start : start + chunk])

                log_probs = -0.5 * (
                    np.log(2 * np.pi * self.sigma2)
                    + (ratings_t[start : start + chunk] - predictions) ** 2 / self.sigma2
                )
                result.append(torch.logsumexp(log_probs, dim=0) - np.log(n_samples))

        return torch.cat(result).cpu().numpy().astype(np.float64)

In [ ]:
# Faça o treinamento e a avaliação do VAE aqui

**Discussão:**
*Escreva suas conclusões aqui*

---

### Visualização dos espaços latentes

Dos modelos treinados, quatro deles aprendem vetores latentes por usuário e por item: PCA com imputação, fatoração de matrizes, PPCA e VAE.
Esses vetores podem ser analisados com as mesmas ferramentas, permitindo que possamos interpretar e comparar os espaços latentes aprendidos por cada modelo.
Garante-se que pelo menos um modelo de cada tipo tenha sido treinado, e então podemos comparar os espaços latentes aprendidos.

> **Tarefa 2.3**
Visualize os espaços latentes de usuários e de itens dos quatro modelos, usando um $k$ de sua escolha, e relacione o que aparece nas figuras com informações que não foram usadas no treino: por exemplo, gênero, idade e ocupação dos usuários, e gêneros, ano e popularidade dos filmes.
>
> O espaço de cada modelo organiza essas informações, e isso pode ser medido além da inspeção visual?
Os quatro espaços compartilham a mesma estrutura, ou cada suposição produz uma geometria diferente?


**Ideias para começar.**

Para visualizar, projete os vetores em duas dimensões com PCA, que é determinístico e preserva a geometria global, e compare com o t-SNE.
Colorir os pontos por um metadado mostra se ele aparece, e um modelo supervisionado linear (`Ridge` para variáveis contínuas, `LogisticRegression` com `roc_auc_score` para categorias) pode medir quanto de informação sobre o metadado está presente no espaço latente original, sem precisar de uma projeção.

Isso permite verificar se, a partir dos vetores latentes, é possível prever a idade do usuário, o gênero do filme ou a popularidade do filme, quantificando o quanto de informação sobre essas variáveis está presente no espaço latente.

Ao comparar os espaços latentes, é importante lembrar que cada modelo aprende uma representação diferente, e que a correspondência entre os vetores latentes de um modelo e de outro não é trivial, pois os espaços latentes podem ser invariantes à rotação, reflexão e escala.
Para comparar dois espaços é necessário alinhar os vetores latentes de um modelo com os do outro, ou através de análises de correspondência de vizinhos, como a correlação de distâncias entre pontos, ou através da sobreposição dos vizinhos mais próximos de cada filme em dois modelos.

In [ ]:
# Faça as visualizações e análises dos espaços latentes aqui

**Discussão:**
*Escreva suas conclusões aqui*

---

## Etapa 3: Projeto do grupo

Nesta última etapa, cada um de vocês deve propor um projeto de investigação, que pode ser uma melhoria do modelo ou uma análise mais profunda de algum aspecto do problema.
Após finalizar as duas primeiras etapas, verifiquem se há alguma pergunta que surgiu durante as análises e ainda não foi explorado, ou alguma hipótese que vocês gostariam de testar, por exemplo, sobre a geometria do espaço latente, uma formulação alternativa, ou um hiperparâmetro que não foi isolado.

O importante é que o projeto esteja bem definido, com uma motivação clara, uma hipótese testável e um protocolo experimental que permita avaliar o impacto da mudança proposta.

> **Tarefa 3.1**
Escolha um projeto e escreva a pergunta, a hipótese, a previsão e o experimento antes de rodar.
Defina o modelo referência, qual é a mudança proposta e qual resultado refutaria a hipótese de vocês.
Ao final, compare o que foi previsto com o que foi observado e o que o resultado diz sobre a previsão sobre o espaço latente.

**Projetos sugeridos.**
Aqui está uma lista de ideias que podem ser exploradas, mas vocês podem propor outras, desde que estejam bem justificadas.

*1. Ausência não aleatória.*
Usuários ativos e filmes populares aparecem mais nos dados, o que cria um viés de observação.
Pergunta: corrigir esse viés com pesos pelo inverso da probabilidade de observação [Schnabel et al.](https://arxiv.org/abs/1602.05352) melhora as previsões para filmes pouco avaliados, e o que acontece com os populares?

*2. Metadados como features.*
Ao invés de tratar cada usuário e item como um identificador único, podemos usar suas características como features para o "codificador".
Pergunta: usar os metadados para definir a média do prior de $\mathbf{z}$ melhora as previsões de usuários com pouco histórico?

*3. Geometria da compatibilidade.*
Todos os modelos utilizam o produto interno para medir a compatibilidade entre usuários e itens.
Pergunta: trocar o produto interno por uma distância, ou por uma pequena rede que recebe $\mathbf{z}_u$ e $\mathbf{w}_i$, muda o que o espaço codifica e o erro de previsão? Ver [Rendle et al.](https://arxiv.org/abs/2005.09683) para um resultado que motiva a pergunta.

*4. Estrutura do espaço latente.*
A selecão de hiperparâmetros não observou a geometria do espaço latente, mas apenas o erro de previsão.
Pergunta: o que muda no espaço latente aprendido do VAE ao variar $\beta$, o tamanho do espaço ou a regularização, em termos de estabilidade de clusters, correlação com metadados e erro de previsão?

In [ ]:
# Desenvolva o experimento do seu grupo aqui

**Discussão:**
*Escreva suas conclusões aqui*